# PH-SHOWOA · Full-GPU Tensorized RCRS-GRASP + SA Benchmark

**Mục tiêu**: Chạy thuật toán PH-SHOWOA Full-GPU Tensorized (`src_cpp_gpu_SA_RCRS_GRASP`) tăng tốc GPU (CUDA) với kiến trúc **Full-GPU + Tensorization** (Branchless Masking, Warp Reduction, Zero-Copy GPU VRAM) và phương pháp khởi tạo kết hợp **RCRS-GRASP + SA Post-refinement** trên 15 bộ dữ liệu VRPSPDTW chuẩn của Wang & Chen.

**Thông số thử nghiệm**:
- `Architecture` = `full_gpu` (100% dữ liệu tiến hóa nằm trên GPU VRAM)
- `Popsize` = 30
- `Max-iteration` = 1000
- `Runs` = 30
- `Init` = `rcrs_grasp` (Tự động kèm 25 vòng SA Post-refinement)

**GPU khuyên dùng trên Kaggle**: **NVIDIA Tesla T4** / **NVIDIA P100**

## Cell 1 – Clone repo từ GitHub

In [ ]:
!git clone https://github.com/Welkie/ph-showoa.git

## Cell 2 – Build C++ Full-GPU Tensorized SA-RCRS-GRASP solver (CUDA)

In [ ]:
%%bash
set -e
cd ph-showoa/src_cpp_gpu_SA_RCRS_GRASP
mkdir -p build && cd build
cmake -DENABLE_CUDA=ON -DCMAKE_BUILD_TYPE=Release .. 2>&1 | tail -10
make -j$(nproc) 2>&1
echo "=== Build Full-GPU Tensorized SA-RCRS-GRASP OK ==="
ls -lh phshowoa_cpp

## Cell 3 – Batch runner Full-GPU Tensorized RCRS-GRASP + SA (15 bộ dữ liệu)

In [ ]:
import os, glob, subprocess, re, sys, csv, time as T
from collections import deque

# ── Cấu hình 15 bộ dữ liệu ───────────────────────────────────────────────────
DATASETS = [
    "rcdp1001", "rcdp5001", "rcdp5007", "rcdp5004", "rcdp101",
    "cdp103",   "rcdp205",  "rdp210",   "rcdp207",  "rcdp202",
    "rdp103",   "cdp104",   "cdp102",   "rdp203",   "rcdp104"
]
CWD          = "ph-showoa/src_cpp_gpu_SA_RCRS_GRASP/build"
BINARY       = "./phshowoa_cpp"
DATASET_DIR  = "/kaggle/input/datasets/keith1101/ph-showoa/Wang_Chen"
OUTPUT_CSV   = "/kaggle/working/summary_full_gpu_tensorized.csv"
RUNS         = 30
MAX_ITER     = 1000
POP_SIZE     = 128 # 30=Hybrid, 128/256=Full-GPU
INIT_MODE    = "rcrs_grasp"
COMPUTE_BACKEND = "cuda"
ARCHITECTURE    = "full_gpu"

# Số dòng output gần nhất in ra màn hình (tránh IOPub flood)
TAIL_LINES   = 8

# ── Helpers ───────────────────────────────────────────────────────────────────
def find_file(name):
    p = os.path.join(DATASET_DIR, f"explicit_{name}.vrpsdptw")
    if os.path.exists(p): return p
    # Tìm không phân biệt hoa thường nếu cần
    found = glob.glob(f"/kaggle/input/**/explicit_{name}.vrpsdptw", recursive=True)
    if not found:
        found = glob.glob(f"/kaggle/input/**/[eE][xX][pP][lL][iI][cC][iI][tT]_{name}.vrpsdptw", recursive=True)
    return found[0] if found else None

def parse_output(text):
    runs_m  = re.search(r"Total (\d+) runs, total consumed (\d+) sec", text)
    nv_m    = re.search(r"vehicle \(route\) number:\s*(\d+)", text)
    cost_m  = re.search(r"Total cost:\s*([\d.]+)", text)
    if runs_m and nv_m and cost_m:
        tr   = int(runs_m.group(1))
        ts   = float(runs_m.group(2))
        nv   = int(nv_m.group(1))
        cost = float(cost_m.group(1))
        td   = cost - 2000.0 * nv
        return {"best_NV": nv, "best_TD": f"{td:.4f}",
                "total_cost": f"{cost:.4f}",
                "avg_time_s": f"{ts/tr:.2f}" if tr else "N/A",
                "total_runs": tr, "Status": "Success"}
    return {"best_NV":"N/A","best_TD":"N/A","total_cost":"N/A",
            "avg_time_s":"N/A","total_runs":0,"Status":"Parse Error"}

# ── Batch loop ────────────────────────────────────────────────────────────────
print("="*70)
print(f"  PH-SHOWOA FULL-GPU TENSORIZED  |  runs={RUNS}  iter={MAX_ITER}  pop={POP_SIZE}  arch={ARCHITECTURE}")
print("="*70)

results = []

for name in DATASETS:
    print(f"\n{'─'*60}")
    print(f"  ▶  {name}")
    print(f"{'─'*60}")

    fp = find_file(name)
    if fp is None:
        print(f"  [SKIP] File not found for: {name}")
        results.append({"Dataset":name,"best_NV":"N/A","best_TD":"N/A",
                        "total_cost":"N/A","avg_time_s":"N/A",
                        "wall_time":"N/A","total_runs":0,"Status":"File Not Found"})
        continue

    # Khởi tạo Full-GPU Tensorized SA-RCRS-GRASP
    cmd = [
        BINARY, fp,
        "--architecture", ARCHITECTURE,
        "--compute_backend", COMPUTE_BACKEND,
        "--paper_flags",
        "--init", INIT_MODE,
        "--runs",     str(RUNS),
        "--max_iter", str(MAX_ITER),
        "--pop_size", str(POP_SIZE),
    ]

    t0 = T.time()

    proc = subprocess.Popen(
        cmd, cwd=CWD,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    all_lines  = []
    tail_buf   = deque(maxlen=TAIL_LINES)
    last_print = T.time()

    for line in proc.stdout:
        all_lines.append(line)
        tail_buf.append(line.rstrip())

        # In status mỗi 30s để duy trì kết nối stdout với Kaggle
        now = T.time()
        if now - last_print >= 30:
            elapsed = now - t0
            print(f"  [{elapsed:.0f}s] running... last: {tail_buf[-1][:80]}",
                  flush=True)
            last_print = now

    proc.wait()
    wall = T.time() - t0

    print(f"  Wall time: {wall:.1f}s")
    print("  --- Last output ---")
    for ln in tail_buf:
        print(" ", ln)

    res = parse_output("".join(all_lines))
    res["Dataset"]   = name
    res["wall_time"] = f"{wall:.1f}s"
    results.append(res)

    print(f"  ✓ best_NV={res['best_NV']}  best_TD={res['best_TD']}  status={res['Status']}")

print(f"\n{'='*70}")
print("  Hoàn tất.")
print('='*70)

## Cell 4 – Bảng tổng hợp kết quả & xuất CSV

In [ ]:
import pandas as pd

cols_order = ["Dataset","best_NV","best_TD","total_cost","avg_time_s","wall_time","total_runs","Status"]
df = pd.DataFrame(results)[cols_order]
df.columns = ["Dataset","Best NV","Best TD","Total Cost","Avg/Run","Wall Time","Runs","Status"]

print(f"Mode: Full-GPU Tensorized SA-RCRS-GRASP | Init: {INIT_MODE} | max_iter={MAX_ITER} | pop={POP_SIZE}\n")
print(df.to_string(index=False))

df.to_csv(OUTPUT_CSV, index=False)
print(f"\n[OK] Saved → {OUTPUT_CSV}")